# 04 · Validate — LigandMPNN vs ProteinMPNN + sequence-specificity analysis

**Standard slot:** *validate (in silico).* **For Project 23 this is the core comparison:** the
head-to-head between **LigandMPNN (NA-aware)** and **ProteinMPNN (NA-blind)** at the interface, the
**sequence-specificity analysis** (intended motif vs scrambled motif), and protein–NA complex-modeling
figures, with publication-style plots (D3 part 2).

The thesis of this notebook: **the value of LigandMPNN is specificity, not just confidence.** ProteinMPNN
can produce confident-looking designs, but because it is blind to the nucleic acid it should be **less
likely to prefer the intended motif** over a scramble. We test exactly that.

Needs `results/ligandmpnn_designs.csv` + `results/proteinmpnn_designs.csv` + `results/all_ranked.csv`
(from notebooks 02–03).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Head-to-head: confidence hit rate **and** specificity rate

Compare the two designers on (a) confidence hit rate (passed the shared layers), (b) **specificity rate**
(confident AND `specificity_score ≥ margin`), and (c) the `pae_interaction` distribution. A fair
comparison filters both identically (notebook 03) and reports the *distribution*, not the single best.
Mock numbers are SYNTHETIC.

In [ ]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
SPEC_MARGIN = 1.5
print("sequence designers:", ranked["seq_tool"].value_counts().to_dict())

summary = []
for t, g in ranked.groupby("seq_tool"):
    n = len(g)
    conf = int((g["layers_passed"] >= 3).sum())
    spec = int(((g["layers_passed"] >= 3) & (g["specificity_score"] >= SPEC_MARGIN)).sum())
    summary.append(dict(seq_tool=t, n=n, confident=conf, confident_specific=spec,
                        confidence_rate_pct=round(100*conf/max(n,1), 1),
                        specificity_rate_pct=round(100*spec/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_dScore=round(float(g["specificity_score"].median()), 2)))
summary = pd.DataFrame(summary)
print("\nhead-to-head summary (SYNTHETIC if mock):")
print(summary.to_string(index=False))
print("\nKEY READOUT: compare the SPECIFICITY rate (NA-aware LigandMPNN should win there if NA conditioning helps).")

In [ ]:
# (a) specificity_score distribution per designer; (b) pae_interaction distribution per designer.
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for t, g in ranked.groupby("seq_tool"):
    ax[0].hist(g["specificity_score"].dropna(), bins=15, alpha=0.5, label=t)
    ax[1].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=t)
ax[0].axvline(SPEC_MARGIN, ls="--", color="k", lw=1, label=f"spec margin ({SPEC_MARGIN})")
ax[0].set_xlabel("specificity_score = score(scrambled) - score(motif)\n(higher = prefers intended motif)")
ax[0].set_ylabel("designs"); ax[0].set_title("Sequence specificity"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("pae_interaction (Å, lower better)"); ax[1].set_title("Protein-NA complex confidence"); ax[1].legend(fontsize=8)
fig.suptitle("LigandMPNN (NA-aware) vs ProteinMPNN (NA-blind) (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p23_headtohead.png", dpi=150); plt.show()
print("saved results/p23_headtohead.png")

## 2 · The specificity analysis, made explicit

This is the scientific heart of the project. For each design we compare the interface score on the
**intended motif** vs a **scrambled motif** (same base composition, different order). A point on the
diagonal is **non-specific** (reads the backbone, not the bases); a point below the diagonal **prefers
the intended motif** (specific). Color by designer to see whether nucleic-acid conditioning (LigandMPNN)
pushes designs off the diagonal. Mock numbers are SYNTHETIC — but the *shape* of this analysis is exactly
what you produce on Colab.

In [ ]:
lig = pd.read_csv("results/ligandmpnn_designs.csv")
pro = pd.read_csv("results/proteinmpnn_designs.csv")
pools = pd.concat([lig, pro], ignore_index=True)

fig, ax = plt.subplots(figsize=(5.2, 5))
for t, g in pools.groupby("seq_tool"):
    ax.scatter(g["dG_motif"], g["dG_scrambled"], s=18, alpha=0.6, label=t)
lo = float(np.nanmin(pools[["dG_motif", "dG_scrambled"]].values))
hi = float(np.nanmax(pools[["dG_motif", "dG_scrambled"]].values))
ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="non-specific (diagonal)")
ax.set_xlabel("interface score vs INTENDED motif (lower = stronger)")
ax.set_ylabel("interface score vs SCRAMBLED motif")
ax.set_title("Specificity: below the diagonal = prefers the intended motif\n(EXAMPLE_DATA if mock)")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("results/p23_specificity.png", dpi=150); plt.show()
print("saved results/p23_specificity.png")
print("Points ON the diagonal are NON-SPECIFIC backbone-grippers — the honest hard truth of protein-NA design.")

## 3 · Novelty `[extension]`

Novelty = TM-score of each protein backbone to its nearest natural fold (Foldseek/TM-align; `< 0.5` ≈
novel). On Colab, compute it per design and compare the two designers' novelty distributions. Here we
scaffold the analysis (mock has no real structures), so we just show where it plugs in.

In [ ]:
# Scaffold: on Colab, run Foldseek/TM-align on each predicted protein backbone -> tm_to_pdb,
# then compare distributions across designers (novel == tm_to_pdb < 0.5).
if "tm_to_pdb" in ranked.columns and ranked["tm_to_pdb"].notna().any():
    for t, g in ranked.groupby("seq_tool"):
        novel = (g["tm_to_pdb"] < 0.5).mean()
        print(f"{t:12s}: novel fraction (TM<0.5) = {novel:.2f}")
else:
    print("Novelty scaffold — populate tm_to_pdb with Foldseek/TM-align on Colab, then compare designers.")

## 4 · CRISPR-modulator framing `[extension]`

A topical extension: instead of (or in addition to) a free nucleic-acid motif, target a **Cas protein
surface** to build a *CRISPR modulator* — a mini-protein that tunes or blocks Cas activity (an
anti-CRISPR-like modulator for safer gene editing). That is a **protein–protein** binder problem
(reuse the Project 06 workflow against a Cas target), optionally combined with NA context if the Cas–
guide–DNA complex is the target. Keep the framing **therapeutic / basic-science** (control/safety of
editing), per Responsible Research. Here we just flag where it plugs in.

In [ ]:
# Scaffold ONLY. CRISPR-modulator extension = a protein-protein binder vs a Cas surface
# (reuse projects/project_06_pdl1_binder/scripts/binder_tools.py against a Cas target PDB),
# optionally with the guide-RNA/target-DNA as NA context (this project's na_binder_tools).
# Responsible Research: frame as editing CONTROL/safety (an anti-CRISPR-like modulator), not harm.
print("CRISPR-modulator framing is an [extension] scaffold: bind a Cas surface to MODULATE editing.")
print("Treat it as a protein-protein binder (Project 06 workflow) +/- NA context; keep the framing therapeutic.")

## 5 · Select the top specific candidates per designer

The D★ deliverable wants designs that are **confident AND specific**. Rank the confident-and-specific
set by the composite score and, as a tie-breaker, prefer a higher `specificity_score`. Save the
shortlist for the validation plan (notebook 05).

In [ ]:
top_per = []
for t, g in ranked.groupby("seq_tool"):
    g2 = g[(g["layers_passed"] >= 3) & (g["specificity_score"] >= SPEC_MARGIN)].sort_values(
        ["score", "specificity_score"], ascending=False).head(15)
    top_per.append(g2)
top = pd.concat(top_per, ignore_index=True) if top_per else pd.DataFrame()
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top confident+specific per designer)")
if len(top):
    print(top.groupby("seq_tool").size().to_dict())
    print(top.head(8)[["design_id", "seq_tool", "score", "pae_interaction", "specificity_score"]].to_string(index=False))
else:
    print("No confident+specific designs in this mock run — that is a legitimate (and common) outcome to report.")

## D3 (part 2) checklist
- [ ] Head-to-head: confidence rate **and** specificity rate per designer (figure `results/p23_headtohead.png`).
- [ ] **Specificity analysis** (intended vs scrambled motif) plotted (`results/p23_specificity.png`); non-specific designs called out.
- [ ] Novelty compared across designers (TM-score to PDB) — or the scaffold wired up on Colab.
- [ ] (Extension) CRISPR-modulator framing noted; (extension) the LigandMPNN-vs-ProteinMPNN interface comparison discussed.
- [ ] `results/top_candidates.csv`: top **confident + specific** designs, ready for the validation plan.
- [ ] Honest discussion: confidence ≠ specificity; many designs grip the backbone non-specifically.

**Next:** `05_validation_plan.ipynb` — the EMSA / fluorescence-anisotropy plan with scrambled-NA controls.